<a href="https://colab.research.google.com/github/Leanhchudang2511/baitaptrituenhantao/blob/main/AppDuToanGiaTienPhongTro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 TÍNH TIỀN PHÒNG TRỌ - APP THÔNG MINH VỚI MACHINE LEARNING
### Quản lý dễ dàng – Thu tiền chính xác – Dự đoán AI (Streamlit + ngrok)

**Chạy từng cell theo thứ tự từ trên xuống dưới**

---
### Các bước:
1. **Cell 1**: Cài thư viện (không còn lỗi websockets!)
2. **Cell 2**: Import & dữ liệu mẫu
3. **Cell 3**: Train ML models & lưu model.pkl
4. **Cell 4**: Viết file app.py (Streamlit)
5. **Cell 5**: Lấy ngrok token → Khởi động → Nhận PUBLIC LINK

In [1]:

!pip install streamlit pyngrok scikit-learn pandas openpyxl matplotlib seaborn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.0 MB/s eta 0:00:00
✅ Cài đặt xong!


In [2]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import os, json, hashlib, warnings, pickle
from datetime import datetime
warnings.filterwarnings('ignore')

EXCEL_FILE = 'giadienphongtro.xlsx'

KHU_VUC = {
    'TP.HCM - Quận 1': 1.5,  'TP.HCM - Quận 2 (Thủ Đức)': 1.35,
    'TP.HCM - Quận 3': 1.45, 'TP.HCM - Quận 7 (Phú Mỹ Hưng)': 1.4,
    'TP.HCM - Bình Thạnh': 1.2, 'TP.HCM - Tân Bình': 1.25,
    'TP.HCM - Gò Vấp': 1.1, 'TP.HCM - Bình Chánh': 0.9,
    'TP.HCM - Hóc Môn': 0.85, 'Hà Nội - Hoàn Kiếm': 1.45,
    'Hà Nội - Cầu Giấy': 1.3, 'Hà Nội - Đống Đa': 1.35,
    'Hà Nội - Hà Đông': 1.0, 'Đà Nẵng': 1.1,
    'Cần Thơ': 0.95, 'Khác': 1.0,
}

SAMPLE = {
    'loai_nguoi': ['Sinh viên','Hộ gia đình','Sinh viên','Hộ gia đình','Sinh viên',
                   'Hộ gia đình','Sinh viên','Hộ gia đình','Sinh viên','Hộ gia đình',
                   'Sinh viên','Hộ gia đình','Sinh viên','Hộ gia đình','Sinh viên',
                   'Hộ gia đình','Sinh viên','Hộ gia đình','Sinh viên','Hộ gia đình'],
    'so_nguoi': [2,3,1,4,3,5,2,5,3,2,2,2,2,2,4,5,5,2,2,1],
    'dien_tich': [23,50,25,53,25,78,20,85,46,101,20,85,25,54,79,114,95,26,17,26],
    'tang': [3,9,4,7,4,4,4,0,19,9,4,8,4,19,1,9,14,0,2,3],
    'loai_hinh': ['Ky tuc xa','Can ho','Phong tro','Can ho','Phong tro',
                  'Nha pho','Phong tro','Can ho','Can ho','Can ho',
                  'Phong tro','Nha pho','Can ho dich vu','Can ho','Nha pho',
                  'Can ho','Nha pho','Phong tro','Phong tro','Phong tro'],
    'co_tu_lanh': [1]*20,
    'so_quat': [2,2,1,3,1,3,2,3,2,3,2,3,2,3,3,4,2,1,2,2],
    'so_may_lanh': [1,3,0,1,1,1,1,3,3,1,1,2,1,2,1,1,1,1,0,0],
    'gio_may_lanh': [4,7,0,13,7,7,13,6,7,10,13,9,11,9,8,5,13,9,0,0],
    'tien_dien': [952000,2052000,448000,1478000,1300000,935000,2240000,2590000,
                  2481000,1722000,2240000,1885000,1843000,2590000,1494000,
                  1050000,1939000,1516000,552000,564000],
    'gia_dien': [3500,2500,3800,2500,3800,2500,4000,3500,3000,3500,
                 4000,2500,3800,3500,3500,3000,3500,3800,4000,4000],
    'khu_vuc': ['TP.HCM - Tân Bình']*20,
    'thang': [datetime.now().strftime('%Y-%m')]*20,
}

def load_data():
    if os.path.exists(EXCEL_FILE):
        try:
            raw = pd.read_excel(EXCEL_FILE)
            if 'tien_dien' in raw.columns:
                df = raw[raw['tien_dien'] > 0].dropna(subset=['tien_dien']).reset_index(drop=True)
                print(f'✅ Load {len(df)} bản ghi từ Excel!')
                return df
            c = raw.columns.tolist()
            df = pd.DataFrame()
            df['loai_nguoi']  = raw[c[1]].astype(str)
            df['so_nguoi']    = pd.to_numeric(raw[c[2]], errors='coerce').fillna(2)
            df['dien_tich']   = pd.to_numeric(raw[c[3]], errors='coerce').fillna(25)
            df['tang']        = pd.to_numeric(raw[c[4]], errors='coerce').fillna(1)
            df['loai_hinh']   = raw[c[5]].astype(str)
            df['co_tu_lanh']  = (raw[c[6]].astype(str).str.strip() == 'Có').astype(int)
            df['so_quat']     = pd.to_numeric(raw[c[7]], errors='coerce').fillna(1)
            df['so_may_lanh'] = pd.to_numeric(raw[c[8]], errors='coerce').fillna(0)
            df['gio_may_lanh']= pd.to_numeric(raw[c[9]], errors='coerce').fillna(0)
            df['tien_dien']   = pd.to_numeric(raw[c[10]], errors='coerce').fillna(500000)
            df['gia_dien']    = pd.to_numeric(raw[c[11]], errors='coerce').fillna(3500)
            df['khu_vuc']     = 'TP.HCM - Tân Bình'
            df['thang']       = datetime.now().strftime('%Y-%m')
            df = df[df['tien_dien'] > 0].dropna(subset=['tien_dien'])
            print(f'✅ Load {len(df)} bản ghi từ Excel!')
            return df
        except Exception as e:
            print(f'⚠️ Lỗi đọc Excel: {e}')
    df = pd.DataFrame(SAMPLE)
    df.to_excel(EXCEL_FILE, index=False)
    print(f'✅ Tạo dữ liệu mẫu {len(df)} bản ghi!')
    return df

df_main = load_data()
print(df_main[['loai_nguoi','dien_tich','tien_dien','gia_dien']].head())
print('✅ Import xong!')

✅ Tạo dữ liệu mẫu 20 bản ghi!
    loai_nguoi  dien_tich  tien_dien  gia_dien
0    Sinh viên         23     952000      3500
1  Hộ gia đình         50    2052000      2500
2    Sinh viên         25     448000      3800
3  Hộ gia đình         53    1478000      2500
4    Sinh viên         25    1300000      3800
✅ Import xong!


In [3]:

def get_features(df, df_ref=None):
    if df_ref is None:
        df_ref = df
    d = df.copy()
    all_nguoi = list(df_ref['loai_nguoi'].unique())
    all_hinh  = list(df_ref['loai_hinh'].unique())
    le_n = LabelEncoder().fit(all_nguoi)
    le_h = LabelEncoder().fit(all_hinh)
    d['loai_nguoi_enc'] = le_n.transform([x if x in all_nguoi else all_nguoi[0] for x in d['loai_nguoi'].astype(str)])
    d['loai_hinh_enc']  = le_h.transform([x if x in all_hinh  else all_hinh[0]  for x in d['loai_hinh'].astype(str)])
    d['kwh_ml']    = d['so_may_lanh'] * d['gio_may_lanh'] * 1.5 * 30
    d['kwh_quat']  = d['so_quat'] * 0.075 * 8 * 30
    d['kwh_tl']    = d['co_tu_lanh'] * 0.1 * 24 * 30
    d['kwh_total'] = d['kwh_ml'] + d['kwh_quat'] + d['kwh_tl'] + 20
    return d[['loai_nguoi_enc','so_nguoi','dien_tich','tang','loai_hinh_enc',
              'co_tu_lanh','so_quat','so_may_lanh','gio_may_lanh','gia_dien',
              'kwh_ml','kwh_quat','kwh_tl','kwh_total']]

def train_all(df):
    X = get_features(df)
    y = df['tien_dien']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    models = {
        'Random Forest':     RandomForestRegressor(n_estimators=150, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, random_state=42),
        'Ridge Regression':  Ridge(alpha=1.0),
    }
    results = {}
    best, best_r2, best_name = None, -999, ''
    print('🤖 Kết quả huấn luyện ML:')
    print('='*55)
    for name, m in models.items():
        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)
        r2   = r2_score(y_te, pred)
        mae  = mean_absolute_error(y_te, pred)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        results[name] = {'model': m, 'r2': r2, 'mae': mae, 'rmse': rmse}
        print(f'  {name}: R²={r2:.4f} | MAE={mae:,.0f}đ')
        if r2 > best_r2:
            best_r2, best, best_name = r2, m, name
    print(f'🏆 Best: {best_name} (R²={best_r2:.4f})')
    return best, results

best_model, model_results = train_all(df_main)

# Lưu model & metadata để Streamlit app dùng
with open('model.pkl', 'wb') as f:
    pickle.dump({
        'model':   best_model,
        'results': model_results,
        'df':      df_main,
        'khu_vuc': KHU_VUC
    }, f)
print('\n✅ Huấn luyện & lưu model.pkl xong!')

🤖 Kết quả huấn luyện ML:
  Random Forest: R²=0.9151 | MAE=116,782đ
  Gradient Boosting: R²=0.9737 | MAE=64,227đ
  Ridge Regression: R²=0.9643 | MAE=65,588đ
🏆 Best: Gradient Boosting (R²=0.9737)

✅ Huấn luyện & lưu model.pkl xong!


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle, os, json, hashlib
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

EXCEL_FILE = 'giadienphongtro.xlsx'
USER_FILE  = 'users.json'

st.set_page_config(page_title='Tinh Tien Phong Tro AI', page_icon='🏠', layout='wide')

st.markdown('''
<style>
.main-header {
  background: linear-gradient(135deg, #1e3a8a, #2563eb);
  color: white; padding: 22px; border-radius: 14px;
  text-align: center; margin-bottom: 18px;
  box-shadow: 0 6px 24px rgba(37,99,235,0.3);
}
.result-box {
  background: #f0f9ff; border: 2px solid #2563eb;
  border-radius: 12px; padding: 16px;
}
</style>
''', unsafe_allow_html=True)

st.markdown('''
<div class="main-header">
  <div style="font-size:42px">🏠</div>
  <h1 style="margin:4px 0;font-size:26px;font-weight:800">TINH TIEN PHONG TRO</h1>
  <p style="opacity:.9;margin:4px 0">🤖 AI Du doan | 📊 Thong ke | 💾 Luu Excel | 📍 Theo Khu vuc</p>
</div>
''', unsafe_allow_html=True)

KHU_VUC = {
    'TP.HCM - Quan 1': 1.5,  'TP.HCM - Quan 2 (Thu Duc)': 1.35,
    'TP.HCM - Quan 3': 1.45, 'TP.HCM - Quan 7 (Phu My Hung)': 1.4,
    'TP.HCM - Binh Thanh': 1.2, 'TP.HCM - Tan Binh': 1.25,
    'TP.HCM - Go Vap': 1.1,  'TP.HCM - Binh Chanh': 0.9,
    'TP.HCM - Hoc Mon': 0.85,'Ha Noi - Hoan Kiem': 1.45,
    'Ha Noi - Cau Giay': 1.3,'Ha Noi - Dong Da': 1.35,
    'Ha Noi - Ha Dong': 1.0, 'Da Nang': 1.1,
    'Can Tho': 0.95,          'Khac': 1.0,
}

def hp(p): return hashlib.sha256(p.encode()).hexdigest()

def load_users():
    if os.path.exists(USER_FILE):
        with open(USER_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'admin': hp('admin123')}

def save_users(u):
    with open(USER_FILE, 'w', encoding='utf-8') as f:
        json.dump(u, f, ensure_ascii=False)

@st.cache_data
def load_data_cached():
    if os.path.exists(EXCEL_FILE):
        try:
            raw = pd.read_excel(EXCEL_FILE)
            if 'tien_dien' in raw.columns:
                return raw[raw['tien_dien'] > 0].dropna(subset=['tien_dien']).reset_index(drop=True)
        except:
            pass
    sample = {
        'loai_nguoi': ['Sinh vien','Ho gia dinh']*10,
        'so_nguoi':   [2,3,1,4,3,5,2,5,3,2,2,2,2,2,4,5,5,2,2,1],
        'dien_tich':  [23,50,25,53,25,78,20,85,46,101,20,85,25,54,79,114,95,26,17,26],
        'tang':       [3,9,4,7,4,4,4,0,19,9,4,8,4,19,1,9,14,0,2,3],
        'loai_hinh':  ['Ky tuc xa','Can ho','Phong tro','Can ho','Phong tro',
                       'Nha pho','Phong tro','Can ho','Can ho','Can ho',
                       'Phong tro','Nha pho','Can ho dich vu','Can ho','Nha pho',
                       'Can ho','Nha pho','Phong tro','Phong tro','Phong tro'],
        'co_tu_lanh': [1]*20,
        'so_quat':      [2,2,1,3,1,3,2,3,2,3,2,3,2,3,3,4,2,1,2,2],
        'so_may_lanh':  [1,3,0,1,1,1,1,3,3,1,1,2,1,2,1,1,1,1,0,0],
        'gio_may_lanh': [4,7,0,13,7,7,13,6,7,10,13,9,11,9,8,5,13,9,0,0],
        'tien_dien':    [952000,2052000,448000,1478000,1300000,935000,2240000,2590000,
                         2481000,1722000,2240000,1885000,1843000,2590000,1494000,
                         1050000,1939000,1516000,552000,564000],
        'gia_dien':     [3500,2500,3800,2500,3800,2500,4000,3500,3000,3500,
                         4000,2500,3800,3500,3500,3000,3500,3800,4000,4000],
        'khu_vuc':      ['TP.HCM - Tan Binh']*20,
        'thang':        [datetime.now().strftime('%Y-%m')]*20,
    }
    df = pd.DataFrame(sample)
    df.to_excel(EXCEL_FILE, index=False)
    return df

def get_features(df, df_ref):
    d = df.copy()
    all_nguoi = list(df_ref['loai_nguoi'].unique())
    all_hinh  = list(df_ref['loai_hinh'].unique())
    le_n = LabelEncoder().fit(all_nguoi)
    le_h = LabelEncoder().fit(all_hinh)
    d['loai_nguoi_enc'] = le_n.transform([x if x in all_nguoi else all_nguoi[0] for x in d['loai_nguoi'].astype(str)])
    d['loai_hinh_enc']  = le_h.transform([x if x in all_hinh  else all_hinh[0]  for x in d['loai_hinh'].astype(str)])
    d['kwh_ml']    = d['so_may_lanh'] * d['gio_may_lanh'] * 1.5 * 30
    d['kwh_quat']  = d['so_quat'] * 0.075 * 8 * 30
    d['kwh_tl']    = d['co_tu_lanh'] * 0.1 * 24 * 30
    d['kwh_total'] = d['kwh_ml'] + d['kwh_quat'] + d['kwh_tl'] + 20
    return d[['loai_nguoi_enc','so_nguoi','dien_tich','tang','loai_hinh_enc',
              'co_tu_lanh','so_quat','so_may_lanh','gio_may_lanh','gia_dien',
              'kwh_ml','kwh_quat','kwh_tl','kwh_total']]

@st.cache_resource
def train_models():
    if os.path.exists('model.pkl'):
        with open('model.pkl', 'rb') as f:
            d = pickle.load(f)
            return d['model'], d['results']
    df = load_data_cached()
    X  = get_features(df, df)
    y  = df['tien_dien']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    mdls = {
        'Random Forest':     RandomForestRegressor(150, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(150, random_state=42),
        'Ridge Regression':  Ridge(1.0),
    }
    results, best, best_r2 = {}, None, -999
    for name, m in mdls.items():
        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)
        r2   = r2_score(y_te, pred)
        results[name] = {'model': m, 'r2': r2,
                         'mae':  mean_absolute_error(y_te, pred),
                         'rmse': np.sqrt(mean_squared_error(y_te, pred))}
        if r2 > best_r2: best_r2, best = r2, m
    return best, results

# ── Session state ─────────────────────────────────────────
if 'logged_in' not in st.session_state: st.session_state.logged_in = False
if 'username'  not in st.session_state: st.session_state.username  = ''
if 'df'        not in st.session_state: st.session_state.df        = load_data_cached()

best_model, model_results = train_models()
users_db = load_users()

if st.session_state.logged_in:
    st.success(f"👤 Da dang nhap: **{st.session_state.username}** | {datetime.now().strftime('%H:%M %d/%m/%Y')}")
else:
    st.info('👋 Vui long dang nhap de dung day du tinh nang.')

tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
    '🔐 Dang nhap', '🤖 Du doan AI', '📊 Thong ke',
    '💾 Du lieu',   '📍 Khu vuc',    '🧠 Hieu suat ML'
])

# ── TAB 1: DANG NHAP ──────────────────────────────────────
with tab1:
    col_l, col_r = st.columns(2)
    with col_l:
        st.subheader('🔑 Dang nhap')
        lu = st.text_input('Ten dang nhap', placeholder='admin', key='lu')
        lp = st.text_input('Mat khau', type='password', placeholder='admin123', key='lp')
        if st.button('🚀 Dang nhap', type='primary'):
            if lu in users_db and users_db[lu] == hp(lp):
                st.session_state.logged_in = True
                st.session_state.username  = lu
                st.success(f'Chao mung {lu}! Dang nhap thanh cong.')
                st.rerun()
            else:
                st.error('Sai ten dang nhap hoac mat khau!')
        st.caption('Demo: admin / admin123')
    with col_r:
        st.subheader('📝 Dang ky moi')
        ru = st.text_input('Ten dang nhap moi', key='ru')
        rp = st.text_input('Mat khau (>=6 ky tu)', type='password', key='rp')
        rc = st.text_input('Xac nhan mat khau',    type='password', key='rc')
        if st.button('Dang ky'):
            if not ru or not rp:  st.error('Nhap day du!')
            elif len(rp) < 6:     st.error('Mat khau >= 6 ky tu!')
            elif rp != rc:        st.error('Mat khau khong khop!')
            elif ru in users_db:  st.error('Ten da ton tai!')
            else:
                users_db[ru] = hp(rp); save_users(users_db)
                st.success(f'Dang ky thanh cong: {ru}')

# ── TAB 2: DU DOAN AI ─────────────────────────────────────
with tab2:
    st.info('Nhap thong tin phong → AI du doan tien dien → Tu dong luu vao Excel.')
    c1, c2, c3 = st.columns(3)
    with c1:
        st.markdown('**Thong tin nguoi o**')
        p_ln = st.radio('Loai nguoi o', ['Sinh vien', 'Ho gia dinh'])
        p_sn = st.slider('So nguoi o', 1, 10, 2)
        p_kv = st.selectbox('Khu vuc', list(KHU_VUC.keys()), index=5)
        p_gd = st.number_input('Gia dien (d/kWh)', value=3800, step=100)
    with c2:
        st.markdown('**Thong tin phong**')
        p_dt = st.slider('Dien tich (m2)', 10, 200, 25)
        p_tg = st.slider('Tang so', 0, 30, 3)
        p_lh = st.selectbox('Loai hinh',
            ['Phong tro','Can ho','Can ho dich vu','Nha pho','Ky tuc xa'])
    with c3:
        st.markdown('**Thiet bi dien**')
        p_tl = st.radio('Tu lanh', ['Co', 'Khong'])
        p_sq = st.slider('So quat', 0, 6, 2)
        p_sm = st.slider('So may lanh', 0, 5, 1)
        p_gm = st.slider('Gio bat may lanh/ngay', 0.0, 24.0, 8.0, 0.5)

    cb1, cb2 = st.columns(2)
    with cb1: predict_btn = st.button('DU DOAN TIEN DIEN', type='primary', use_container_width=True)
    with cb2: retrain_btn = st.button('Huan luyen lai ML', use_container_width=True)

    if predict_btn:
        try:
            tl      = 1 if p_tl == 'Co' else 0
            kv_mult = KHU_VUC.get(p_kv, 1.0)
            df_ref  = st.session_state.df
            row = pd.DataFrame([{
                'loai_nguoi': p_ln, 'so_nguoi': p_sn, 'dien_tich': float(p_dt),
                'tang': p_tg, 'loai_hinh': p_lh, 'co_tu_lanh': tl,
                'so_quat': p_sq, 'so_may_lanh': p_sm,
                'gio_may_lanh': float(p_gm), 'gia_dien': float(p_gd)
            }])
            Xrow      = get_features(row, df_ref)
            pred_kv   = best_model.predict(Xrow)[0] * kv_mult
            kwh_total = Xrow['kwh_total'].values[0]
            kwh_ml_v  = Xrow['kwh_ml'].values[0]
            kwh_q     = Xrow['kwh_quat'].values[0]
            kwh_tl_v  = Xrow['kwh_tl'].values[0]
            tay       = kwh_total * p_gd

            m1, m2, m3, m4 = st.columns(4)
            m1.metric('Du doan ML',      f'{pred_kv:,.0f} d')
            m2.metric('Tinh tay',        f'{tay:,.0f} d')
            m3.metric('kWh uoc tinh',    f'{kwh_total:,.1f}')
            m4.metric('He so khu vuc',   f'x{kv_mult:.2f}')

            detail = (
                f'<div class="result-box">'
                f'<b>Chi tiet tieu thu dien:</b><br>'
                f'May lanh ({p_sm} cai x {p_gm}h/ngay): <b>{kwh_ml_v:,.1f} kWh</b><br>'
                f'Quat ({p_sq} cai x 8h/ngay): <b>{kwh_q:,.1f} kWh</b><br>'
                f'Tu lanh (24h/ngay): <b>{kwh_tl_v:,.1f} kWh</b><br>'
                f'Thiet bi khac: <b>20.0 kWh</b>'
                f'</div>'
            )
            st.markdown(detail, unsafe_allow_html=True)

            new_row = {
                'loai_nguoi': p_ln, 'so_nguoi': p_sn, 'dien_tich': float(p_dt),
                'tang': p_tg, 'loai_hinh': p_lh, 'co_tu_lanh': tl,
                'so_quat': p_sq, 'so_may_lanh': p_sm, 'gio_may_lanh': float(p_gm),
                'tien_dien': pred_kv, 'gia_dien': float(p_gd),
                'khu_vuc': p_kv, 'thang': datetime.now().strftime('%Y-%m')
            }
            st.session_state.df = pd.concat(
                [df_ref, pd.DataFrame([new_row])], ignore_index=True)
            st.session_state.df.to_excel(EXCEL_FILE, index=False)
            st.caption(f"Da luu | Tong: {len(st.session_state.df)} ban ghi")
        except Exception as e:
            st.error(f'Loi: {e}')

    if retrain_btn:
        st.cache_resource.clear()
        st.success('Da xoa cache — tai lai trang de huan luyen lai!')
        st.rerun()

# ── TAB 3: THONG KE ───────────────────────────────────────
with tab3:
    df = st.session_state.df
    c1, c2, c3, c4 = st.columns(4)
    c1.metric('Tong ban ghi',  len(df))
    c2.metric('TB tien dien',  f"{df['tien_dien'].mean():,.0f} d")
    c3.metric('TB dien tich',  f"{df['dien_tich'].mean():,.1f} m2")
    c4.metric('TB so nguoi',   f"{df['so_nguoi'].mean():,.1f}")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('Phan Tich Tien Dien Phong Tro', fontsize=13, fontweight='bold')
    axes[0,0].hist(df['tien_dien']/1000, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0,0].set_title('Phan bo Tien dien'); axes[0,0].set_xlabel('Nghin VND'); axes[0,0].grid(alpha=0.3)
    lh_list = df['loai_hinh'].unique()
    bp = axes[0,1].boxplot(
        [df[df['loai_hinh']==lh]['tien_dien'].values/1000 for lh in lh_list],
        labels=lh_list, patch_artist=True)
    box_colors = ['steelblue','skyblue','mediumseagreen','sandybrown','tomato']
    for patch, c in zip(bp['boxes'], box_colors[:len(lh_list)]):
        patch.set_facecolor(c)
    axes[0,1].set_title('Theo Loai hinh'); axes[0,1].tick_params(axis='x', rotation=20, labelsize=8)
    for ln, col in [('Sinh vien','steelblue'),('Ho gia dinh','tomato')]:
        m = df['loai_nguoi'] == ln
        if m.sum() > 0:
            axes[1,0].scatter(df[m]['dien_tich'], df[m]['tien_dien']/1000,
                              c=col, label=ln, alpha=0.6, s=40)
    axes[1,0].set_title('Dien tich vs Tien dien'); axes[1,0].set_xlabel('m2'); axes[1,0].legend()
    by_kv = df.groupby('khu_vuc')['tien_dien'].mean().sort_values(ascending=False).head(8)
    axes[1,1].barh(by_kv.index, by_kv.values/1000, color='steelblue', alpha=0.8)
    axes[1,1].set_title('TB Tien dien theo Khu vuc'); axes[1,1].set_xlabel('Nghin VND')
    plt.tight_layout()
    st.pyplot(fig); plt.close(fig)

# ── TAB 4: DU LIEU ────────────────────────────────────────
with tab4:
    st.subheader('Xem & Xuat du lieu')
    df_show = st.session_state.df
    st.dataframe(df_show, use_container_width=True)
    st.caption(f"Tong: {len(df_show)} ban ghi | {datetime.now().strftime('%H:%M %d/%m/%Y')}")
    buf = df_show.to_csv(index=False).encode('utf-8-sig')
    st.download_button('Xuat CSV', buf, 'giadienphongtro.csv', 'text/csv', use_container_width=True)
    uploaded = st.file_uploader('Upload file Excel du lieu that', type=['xlsx'])
    if uploaded:
        with open(EXCEL_FILE, 'wb') as f: f.write(uploaded.getbuffer())
        st.cache_data.clear()
        st.success('Da upload! Tai lai trang de ap dung.')
        st.rerun()
with tab5:
    st.subheader('He so nhan gia dien theo khu vuc')
    kv_df = pd.DataFrame.from_dict(KHU_VUC, orient='index', columns=['He so'])
    kv_df.index.name = 'Khu vuc'
    kv_df = kv_df.sort_values('He so', ascending=False)
    st.dataframe(kv_df.style.background_gradient(cmap='Blues'), use_container_width=True)
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.barh(kv_df.index, kv_df['He so'], color='steelblue', alpha=0.8)
    ax2.axvline(1.0, color='red', linestyle='--', label='Chuan = 1.0')
    ax2.set_xlabel('He so nhan'); ax2.set_title('He so gia dien theo khu vuc'); ax2.legend()
    plt.tight_layout(); st.pyplot(fig2); plt.close(fig2)
with tab6:
    st.subheader('So sanh cac mo hinh Machine Learning')
    rows = []
    for name, res in model_results.items():
        rows.append({'Mo hinh': name,
                     'R2 Score': f"{res['r2']:.4f}",
                     'MAE (d)':  f"{res['mae']:,.0f}",
                     'RMSE (d)': f"{res['rmse']:,.0f}"})
    st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)
    st.caption('R2 >= 0.9: Rat tot | 0.7-0.9: Tot | < 0.7: Can them du lieu')
    st.info(f"Du lieu: {len(st.session_state.df)} ban ghi. Them du lieu → Huan luyen lai → Mo hinh tot hon!")

Writing app.py


In [5]:
NGROK_TOKEN = '3DFvCJVYwIGdAyqwiLEDuaBONez_29crAqnArhG8sbtsvXpC3'

from pyngrok import ngrok, conf
import subprocess, time, threading

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

print('⏳ Đang khởi động Streamlit...')
time.sleep(5)

public_url = ngrok.connect(8501)
print()
print('=' * 55)
print('🚀 APP ĐÃ KHỞI ĐỘNG THÀNH CÔNG!')
print('=' * 55)
print(f'🌐 PUBLIC LINK: {public_url}')
print()
print('📱 Copy link trên → mở trên điện thoại hoặc máy khác!')
print('🔑 Demo login: admin / admin123')
print('⚠️  Link còn hoạt động khi cell này đang chạy.')
print('=' * 55)

⏳ Đang khởi động Streamlit...

🚀 APP ĐÃ KHỞI ĐỘNG THÀNH CÔNG!
🌐 PUBLIC LINK: NgrokTunnel: "https://bruising-slideshow-chapped.ngrok-free.dev" -> "http://localhost:8501"

📱 Copy link trên → mở trên điện thoại hoặc máy khác!
🔑 Demo login: admin / admin123
⚠️  Link còn hoạt động khi cell này đang chạy.
